In [1]:
!pip install torch torchvision

In [17]:
#Import Libraries

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_large
from torch.utils.data import DataLoader

from tqdm import tqdm
import os

In [16]:
#Check GPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [15]:
#Dataset Path

train_path = "/kaggle/input/datasets/cashutosh/gender-classification-dataset/Training"
val_path   = "/kaggle/input/datasets/cashutosh/gender-classification-dataset/Validation"

In [14]:
#Image Transformations

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [13]:
#Load Dataset

train_data = datasets.ImageFolder(train_path, transform=train_transform)
val_data   = datasets.ImageFolder(val_path, transform=val_transform)

print(train_data.classes)

['female', 'male']


In [12]:
#DataLoaders

train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_data,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [11]:
#Load Pretrained MobileNetV3

model = mobilenet_v3_large(weights="DEFAULT")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 211MB/s]


In [18]:
#Freeze Backbone (Feature Extraction)

for param in model.features.parameters():
    param.requires_grad = False

In [19]:
#Replace Final Classifier

model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)

In [20]:
#Move Model to GPU

model = model.to(device)

In [23]:
#Loss and Optimizer

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

In [24]:
#Training Loop

epochs = 5

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"Epoch {epoch+1} Train Accuracy:{acc:.2f}%")

100%|██████████| 735/735 [02:15<00:00,  5.44it/s]


Epoch 1 Train Accuracy:87.61%


100%|██████████| 735/735 [01:44<00:00,  7.05it/s]


Epoch 2 Train Accuracy:88.87%


100%|██████████| 735/735 [01:36<00:00,  7.62it/s]


Epoch 3 Train Accuracy:89.47%


100%|██████████| 735/735 [01:36<00:00,  7.61it/s]


Epoch 4 Train Accuracy:89.96%


100%|██████████| 735/735 [01:35<00:00,  7.67it/s]

Epoch 5 Train Accuracy:90.11%


In [25]:
#Fine-Tuning

for param in model.features[12:].parameters():
    param.requires_grad = True

In [26]:
#Lower learning rate.

optimizer = optim.Adam(model.parameters(), lr=0.00001)

In [27]:
#Fine-Tune Training

epochs = 10

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in tqdm(train_loader):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total

    print(f"FineTune Epoch {epoch+1} Accuracy:{acc:.2f}%")

100%|██████████| 735/735 [01:46<00:00,  6.89it/s]


FineTune Epoch 1 Accuracy:92.79%


100%|██████████| 735/735 [01:39<00:00,  7.38it/s]


FineTune Epoch 2 Accuracy:94.11%


100%|██████████| 735/735 [01:38<00:00,  7.46it/s]


...

FineTune Epoch 8 Accuracy:96.34%


100%|██████████| 735/735 [01:53<00:00,  6.50it/s]


FineTune Epoch 9 Accuracy:96.54%


100%|██████████| 735/735 [01:38<00:00,  7.47it/s]

FineTune Epoch 10 Accuracy:96.62%


In [28]:
#Validation Accuracy

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:",100*correct/total)

Validation Accuracy: 96.66924199502103


In [30]:
torch.save(model.state_dict(),
           "/kaggle/working/gender_mobilenetv3_model.pth")

In [32]:
import os
os.listdir("/kaggle/working")

['.virtual_documents', 'gender_mobilenetv3_model.pth']

In [33]:
from IPython.display import FileLink
FileLink('/kaggle/working/gender_mobilenetv3_model.pth')

/kaggle/working/gender_mobilenetv3_model.pth

In [35]:
!zip -r gender_model.zip /kaggle/working/gender_mobilenetv3_model.pth

  adding: kaggle/working/gender_mobilenetv3_model.pth (deflated 8%)


In [36]:
from IPython.display import FileLink
FileLink("gender_model.zip")

/kaggle/working/gender_model.zip